# Realized-volatility forecasting — ModernTCN vs HAR-RV

Trains **ModernTCN** at h = 1, 5, 22 on `data/EURUSD-RV.csv`, runs the **HAR-RV**
baseline on the same data, and prints a side-by-side comparison table.

Both models share one target and one test set:

$$Y_t^{(h)} \;=\; \ln\!\Big(\tfrac{1}{h}\sum_{k=1}^{h} RV_{t+k}\Big)$$

**Split** (train `year <= 2021` · val `2022-2023` · test `year >= 2024`; HAR folds
val into train since OLS has nothing to tune). Test rows are identical across
models — 647 / 643 / 626 at h = 1 / 5 / 22.

**Runtime:** `Runtime -> Change runtime type -> GPU` (T4 is plenty). A full run is
roughly 10-20 min on GPU. Set `ITR = 1` in the config cell for a ~3 min smoke test.

## 1 · Setup

In [ ]:
import os, subprocess, sys, textwrap

REPO   = "https://github.com/Mr0022/ProjectA.git"
BRANCH = "claude/optimistic-mccarthy-6zszk7"
DIR    = "/content/ProjectA"

if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(DIR)
sys.path.insert(0, DIR)

head = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("HEAD:", head)

# --- guard: refuse to run stale code -----------------------------------------
# The aggregated target must be log(mean RV) == logsumexp(ln_RV) - log(h).
# An older checkout used log(sum RV), which leaves an un-learned ln(h) offset and
# silently inflates MSE (badly so at h=22, where ln(22)^2 = 9.6 dominates the loss).
src = open("exp/exp_ModernTCN.py").read()
assert "math.log(h)" in src and "torch.logsumexp" in src, (
    "This checkout predates the log(mean RV) target. Re-run this cell, or delete "
    f"{DIR} and run it again.")
print("target check: OK  (log(mean RV) = logsumexp(ln_RV) - log(h))")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# Colab already ships torch / pandas / numpy / statsmodels / scipy / scikit-learn.
# This only fills gaps on a bare runtime.
import importlib, subprocess, sys
missing = [p for p, m in [("pandas","pandas"), ("numpy","numpy"), ("statsmodels","statsmodels"),
                          ("scipy","scipy"), ("scikit-learn","sklearn"), ("matplotlib","matplotlib")]
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("dependencies ready" + (f" (installed: {', '.join(missing)})" if missing else ""))

## 2 · Configuration

In [ ]:
EPOCHS   = 40     # max epochs per seed
PATIENCE = 8      # early-stopping patience on validation loss
ITR      = 5      # seeds per horizon: 2021..2021+ITR-1. Set 1 for a quick smoke test.
WORKERS  = 2      # DataLoader workers

HORIZONS = [1, 5, 22]

# Optuna-best ModernTCN hyper-parameters per horizon
# (tuningresults/ModernTCN{1,5,22}/best_params.json, mirrored in scripts/moderntcn.sh).
#
# NOTE: these were tuned on the PREVIOUS dataset and target. They run fine and are a
# reasonable starting point, but re-run tune.py before treating the table as final.
MODERNTCN = {
    1:  dict(seq_len=70, patch_size=16, patch_stride=8,  ffn_ratio=2, num_blocks=2,
             large_size=27, small_size=5, dim=32,
             dropout=0.33157505058759384, head_dropout=0.13413677333143775,
             learning_rate=0.0063484758647924695),
    5:  dict(seq_len=22, patch_size=16, patch_stride=8,  ffn_ratio=1, num_blocks=1,
             large_size=13, small_size=3, dim=128,
             dropout=0.4744918542935045, head_dropout=0.16435589854180058,
             learning_rate=9.048320833685613e-05),
    22: dict(seq_len=22, patch_size=16, patch_stride=2,  ffn_ratio=3, num_blocks=1,
             large_size=51, small_size=7, dim=256,
             dropout=0.3222005482423507, head_dropout=0.18550313066719137,
             learning_rate=0.0001385051157761346),
}
print(f"{len(HORIZONS)} horizons x {ITR} seed(s), up to {EPOCHS} epochs each")

In [ ]:
import re, subprocess, sys

_MEAN   = re.compile(r"^\s*mean\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s*$", re.M)
_STD    = re.compile(r"^\s*std\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s*$", re.M)
_SINGLE = re.compile(r"mse:\s*([-\d.eE+]+),\s*mae:\s*([-\d.eE+]+),"
                     r"\s*rse:\s*([-\d.eE+]+),\s*qlike:\s*([-\d.eE+]+)")
KEYS = ["mse", "mae", "rse", "qlike"]


def run_stream(cmd, show=r"^(train |val |test |Epoch:|>>>>>>> run|mse:|\s*(seed|mean|std)\s|Early)"):
    """Run a command, echo only the interesting lines, return the full output."""
    pat, lines = re.compile(show), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        lines.append(line)
        if pat.search(line):
            print(line.rstrip())
    proc.wait()
    out = "".join(lines)
    if proc.returncode != 0:
        print("\n".join(out.splitlines()[-30:]))
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(map(str, cmd))}")
    return out


def parse_metrics(out):
    """Return {metric: value} and {metric: std}. std is None for a single run."""
    m, s = _MEAN.search(out), _STD.search(out)
    if m:
        mean = {k: float(v) for k, v in zip(KEYS, m.groups())}
        std = {k: float(v) for k, v in zip(KEYS, s.groups())} if s else None
        return mean, std
    hits = _SINGLE.findall(out)
    if not hits:
        raise RuntimeError("could not find metrics in the run output")
    return {k: float(v) for k, v in zip(KEYS, hits[-1])}, None


def moderntcn_cmd(h, p):
    rep = lambda v: [str(v)] * 4          # tune.py expands a single value to 4 stages
    return [sys.executable, "run.py", "--is_training", "1",
            "--model_id", f"ModernTCN_h{h}", "--model", "ModernTCN",
            "--features", "S", "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
            "--aggregate_mean", "--seq_len", str(p["seq_len"]), "--pred_len", str(h),
            "--patch_size", str(p["patch_size"]), "--patch_stride", str(p["patch_stride"]),
            "--ffn_ratio", str(p["ffn_ratio"]),
            "--num_blocks", *rep(p["num_blocks"]),
            "--large_size", *rep(p["large_size"]), "--small_size", *rep(p["small_size"]),
            "--dims", *rep(p["dim"]), "--dw_dims", *rep(p["dim"]),
            "--dropout", str(p["dropout"]), "--head_dropout", str(p["head_dropout"]),
            "--revin", "1", "--use_multi_scale", "False", "--lradj", "TST", "--pct_start", "0.3",
            "--learning_rate", str(p["learning_rate"]), "--batch_size", "256",
            "--train_epochs", str(EPOCHS), "--patience", str(PATIENCE),
            "--num_workers", str(WORKERS), "--itr", str(ITR)]

print("helpers ready")

## 3 · Split sanity check

Confirms every model scores the **same** test rows before anything is trained.

In [ ]:
print(run_stream([sys.executable, "check_splits.py"], show=r".").strip()[-1200:])

## 4 · HAR-RV baseline

OLS, no hyper-parameters, so validation folds into train. Seconds to run, and it
writes figures + CSVs into `HAR-RV results/`.

In [ ]:
import pandas as pd

run_stream([sys.executable, "HAR_RV_run.py"], show=r"^(  (Train|Test|Set)|\s+(MSE|MAE|QLIKE)\s)")

har = pd.read_csv("HAR-RV results/har_rv_all_metrics.csv")
har = har[har["split"] == "test"].set_index("horizon")[["MSE", "MAE", "QLIKE"]]
print("\nHAR-RV test metrics")
display(har.round(4))

## 5 · ModernTCN at h = 1, 5, 22

Each horizon trains `ITR` seeds (2021, 2022, ...) and reports the mean across them.

In [ ]:
import time

mtcn, mtcn_std = {}, {}
for h in HORIZONS:
    print("\n" + "=" * 78)
    print(f"ModernTCN   h = {h}   (seq_len {MODERNTCN[h]['seq_len']}, {ITR} seed(s))")
    print("=" * 78)
    t0 = time.time()
    out = run_stream(moderntcn_cmd(h, MODERNTCN[h]))
    mean, std = parse_metrics(out)
    mtcn[h], mtcn_std[h] = mean, std
    print(f"--> h={h}: MSE {mean['mse']:.4f}  MAE {mean['mae']:.4f}  QLIKE {mean['qlike']:.4f}"
          f"   ({time.time() - t0:.0f}s)")

## 6 · Comparison table

In [ ]:
import numpy as np, pandas as pd

rows = []
for h in HORIZONS:
    rows.append(dict(model="HAR-RV", horizon=h,
                     MSE=har.loc[h, "MSE"], MAE=har.loc[h, "MAE"], QLIKE=har.loc[h, "QLIKE"],
                     MSE_std=np.nan, MAE_std=np.nan, QLIKE_std=np.nan))
    s = mtcn_std[h]
    rows.append(dict(model="ModernTCN", horizon=h,
                     MSE=mtcn[h]["mse"], MAE=mtcn[h]["mae"], QLIKE=mtcn[h]["qlike"],
                     MSE_std=s["mse"] if s else np.nan,
                     MAE_std=s["mae"] if s else np.nan,
                     QLIKE_std=s["qlike"] if s else np.nan))
long = pd.DataFrame(rows)

pd.set_option("display.width", 200)

# headline table: one row per model, horizon x metric across the columns
wide = long.pivot(index="model", columns="horizon", values=["MSE", "MAE", "QLIKE"])
wide = wide[[(m, h) for h in HORIZONS for m in ("MSE", "MAE", "QLIKE")]]   # keep metric order
wide.columns = pd.MultiIndex.from_tuples([(f"h={h}", m) for m, h in wide.columns])
wide = wide.reindex(["HAR-RV", "ModernTCN"])
print("Test-set losses  (lower is better; target = ln(mean RV), identical test rows)\n")
display(wide.round(4))

In [ ]:
# ModernTCN relative to the HAR-RV baseline: negative = ModernTCN wins
delta = pd.DataFrame({
    m: [(mtcn[h][m.lower()] / har.loc[h, m] - 1) * 100 for h in HORIZONS]
    for m in ["MSE", "MAE", "QLIKE"]
}, index=[f"h={h}" for h in HORIZONS])

detail = long.copy()
detail["seeds"] = np.where(detail.model == "ModernTCN", ITR, 1)

print("ModernTCN vs HAR-RV, % change in loss  (negative = ModernTCN better)\n")
display(delta.round(1))
print("\nPer-run detail (std is across seeds; HAR-RV is deterministic OLS)\n")
display(detail.set_index(["horizon", "model"]).round(4))

In [ ]:
# Save + download
long.to_csv("comparison_moderntcn_vs_har.csv", index=False)

md_lines = ["| Model | " + " | ".join(f"h={h} {m}" for h in HORIZONS
                                      for m in ("MSE", "MAE", "QLIKE")) + " |",
            "|" + "---|" * (1 + 3 * len(HORIZONS))]
for model in ["HAR-RV", "ModernTCN"]:
    cells_ = [model]
    for h in HORIZONS:
        r = long[(long.model == model) & (long.horizon == h)].iloc[0]
        cells_ += [f"{r.MSE:.4f}", f"{r.MAE:.4f}", f"{r.QLIKE:.4f}"]
    md_lines.append("| " + " | ".join(cells_) + " |")
table_md = "\n".join(md_lines)
open("comparison_moderntcn_vs_har.md", "w").write(table_md + "\n")
print(table_md)

try:
    from google.colab import files
    files.download("comparison_moderntcn_vs_har.csv")
    files.download("comparison_moderntcn_vs_har.md")
except Exception as e:
    print(f"\n(saved to {os.getcwd()}; auto-download unavailable: {type(e).__name__})")

---

### Reading the numbers

- **MSE / MAE** are on the `ln(mean RV)` scale. **QLIKE** (Patton 2011) exponentiates
  both sides, so it compares variances and is the more robust ranking criterion for
  volatility forecasts.
- All models score the **same** test rows (647 / 643 / 626), so the columns are
  directly comparable. `check_splits.py` in section 3 asserts this.
- HAR-RV is deterministic OLS — one run, no seed spread. ModernTCN's `*_std` columns
  are across seeds; if they are large relative to the gap, the ranking is not solid.

### Things worth knowing

- The ModernTCN hyper-parameters were tuned on the **previous** dataset and target.
  Re-run `tune.py` for a fair "best vs best" comparison.
- To add the LSTM baseline, the tuned commands are in `scripts/lstm.sh` — the same
  `run_stream` / `parse_metrics` helpers work on `LSTM_run.py` output.
- HAR-Q is deliberately absent: it needs realized quarticity, which `EURUSD-RV.csv`
  does not carry, so it runs on a different series and is not comparable here.